# Kaggle Submission Notebook

Loads a saved adapter from an attached Kaggle dataset, runs inference on the competition test set, and writes `submission.csv`.

**Before running:** attach
- the competition dataset
- the SmolVLM model dataset named `smolvlm-500m-instruct` if you want offline-safe loading
- the exported adapter dataset created from the training notebook output


## 0. Install libraries


In [ ]:
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow


## 1. Imports and path discovery

This auto-detects the competition CSVs and looks for an attached adapter folder containing `adapter_config.json` and `adapter_model.safetensors`.


In [ ]:
import glob
import json
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

_kaggle_input = Path('/kaggle/input')
if _kaggle_input.exists():
    print('Contents of /kaggle/input:')
    for _entry in sorted(_kaggle_input.rglob('*'))[:40]:
        print(f'  {_entry}')

    _matches = glob.glob('/kaggle/input/**/test.csv', recursive=True)
    print(f'\nFound test.csv at: {_matches}')
    if not _matches:
        raise RuntimeError(
            'test.csv not found under /kaggle/input. '
            'Make sure the competition dataset is attached to this notebook.'
        )
    DATA_DIR = Path(_matches[0]).parent
    print(f'DATA_DIR set to: {DATA_DIR}')
else:
    DATA_DIR = Path('data')

_HF_MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
_local_model = Path('/kaggle/input/smolvlm-500m-instruct')
MODEL_ID = str(_local_model) if _local_model.exists() else _HF_MODEL_ID
print(f'Model source: {MODEL_ID}')

_adapter_matches = [
    Path(p).parent for p in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
    if (Path(p).parent / 'adapter_model.safetensors').exists()
]
if not _adapter_matches:
    raise RuntimeError(
        'No adapter folder found under /kaggle/input. '
        'Attach a dataset that contains adapter_run8/ with adapter_config.json and adapter_model.safetensors.'
    )
ADAPTER_PATH = _adapter_matches[0]
print(f'Adapter path: {ADAPTER_PATH}')

IMG_SIZE = 224
MAX_CONTEXT_CHARS = 400
CHOICE_LETTERS = 'ABCDEFGHIJ'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2. Load test data


In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')
test_df['choices'] = test_df['choices'].apply(json.loads)
print(f'Test rows: {len(test_df):,}')


## 3. Load base model and adapter

The processor is loaded from the adapter folder so the saved tokenizer and processor files stay paired with the trained adapter.


In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading base model...')
base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
    low_cpu_mem_usage=True,
)

print(f'Loading adapter from {ADAPTER_PATH}...')
model = PeftModel.from_pretrained(base, str(ADAPTER_PATH))
model.eval()

processor = AutoProcessor.from_pretrained(str(ADAPTER_PATH))
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print('Model ready.')


## 4. Prompt and inference helpers

This matches the starter notebook inference logic: single forward pass, then choose the highest log-probability answer letter at the `Answer:` position.


In [ ]:
def _resolve_image_path(data_dir: Path, rel_path: str) -> Path:
    parts = Path(rel_path).parts
    candidates = [
        data_dir / rel_path,
        data_dir / 'images' / rel_path,
        data_dir / Path(rel_path).name,
        data_dir / 'images' / Path(rel_path).name,
    ]
    if parts[0] == 'images' and len(parts) > 1:
        candidates.append(data_dir / Path(*parts[1:]))
        candidates.append(data_dir / 'images' / Path(*parts[1:]))

    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError('Image not found. Tried:\n' + '\n'.join(f'  {p}' for p in candidates))


_LETTER_TOKEN_IDS = {
    letter: processor.tokenizer.encode(f' {letter}', add_special_tokens=False)[-1]
    for letter in CHOICE_LETTERS
}


def build_prompt(row: pd.Series) -> str:
    context_parts = []
    lecture = row.get('lecture', '')
    hint = row.get('hint', '')
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip()[:MAX_CONTEXT_CHARS])
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip()[:MAX_CONTEXT_CHARS])
    context_str = '\n'.join(context_parts)

    choices_str = '\n'.join(
        f'  {CHOICE_LETTERS[i]}. {c}' for i, c in enumerate(row['choices'])
    )

    prompt = '<image>\n'
    if context_str:
        prompt += f'Context:\n{context_str}\n\n'
    prompt += f'Question: {row["question"]}\n'
    prompt += f'Choices:\n{choices_str}\n'
    prompt += 'Answer:'
    return prompt


def predict(row: pd.Series) -> int:
    img = Image.open(_resolve_image_path(DATA_DIR, row['image_path'])).convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
    prompt = build_prompt(row)
    inputs = processor(text=[prompt], images=[img], return_tensors='pt')
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits

    log_probs = F.log_softmax(logits[0, -1, :], dim=-1)
    scores = [log_probs[_LETTER_TOKEN_IDS[CHOICE_LETTERS[i]]].item() for i in range(int(row['num_choices']))]
    return int(torch.tensor(scores).argmax().item())

print(_resolve_image_path(DATA_DIR, test_df.iloc[0]['image_path']))
print('Functions defined.')


## 5. Generate submission.csv

Writes the final file to `/kaggle/working/submission.csv`.


In [ ]:
print('Running inference on test set...')
preds = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test inference'):
    preds.append(predict(row))

sub = pd.DataFrame({'id': test_df['id'].values, 'answer': preds})
output_path = Path('/kaggle/working/submission.csv')
sub.to_csv(output_path, index=False)

print(f'\nDone. {len(sub)} predictions.')
print(f'Saved: {output_path}')
print(sub.head())
print('\nAnswer distribution:')
print(sub['answer'].value_counts().sort_index())


## 6. Next step

Use Kaggle's submission UI to upload `/kaggle/working/submission.csv`.
